# 01 — Exploratory Data Analysis

This notebook explores the **CreditBridge synthetic applicant profiles** dataset.

Goals:
- Understand the distribution of all 21 raw columns
- Examine class imbalance in the `default_label` target
- Inspect demographic splits (gender × geography × income proxy)
- Visualise UPI, utility, mobile, and GST signal patterns
- Check for missing values and schema integrity

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams.update({
    'figure.facecolor': '#F5F0E8',
    'axes.facecolor': '#EDE7D9',
    'axes.edgecolor': '#0A0A0A',
    'axes.labelcolor': '#0A0A0A',
    'xtick.color': '#0A0A0A',
    'ytick.color': '#0A0A0A',
    'font.family': 'monospace',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

PROFILE_PATH = '../data/synthetic/profiles.parquet'
df = pd.read_parquet(PROFILE_PATH)
print(f'Loaded {len(df):,} profiles with {len(df.columns)} columns')
df.head(3)

## 1. Schema & Null Audit

In [ ]:
null_counts = df.isnull().sum()
print('Column dtypes and null counts:')
pd.DataFrame({'dtype': df.dtypes, 'null_count': null_counts, 'null_pct': null_counts / len(df) * 100}).round(2)

## 2. Target Distribution — Default Label

In [ ]:
default_rate = df['default_label'].mean()
print(f'Default rate: {default_rate:.2%}')
print(f'Defaulters   : {df["default_label"].sum():,}')
print(f'Non-defaulters: {(1 - df["default_label"]).sum():,}')

fig, ax = plt.subplots(figsize=(5, 4))
counts = df['default_label'].value_counts()
bars = ax.bar(['Non-Default (0)', 'Default (1)'], counts.values,
              color=['#0066FF', '#D50000'], edgecolor='#0A0A0A', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Default Label Distribution', fontweight='bold', fontsize=13)
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../models/eda_target_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Demographic Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, title in zip(axes,
    ['gender', 'geography', 'income_proxy'],
    ['Gender Split', 'Geography Split', 'Income Proxy Split']):
    vc = df[col].value_counts()
    ax.bar(vc.index, vc.values, color='#0066FF', edgecolor='#0A0A0A', linewidth=1.5)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('../models/eda_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Default Rate by Demographic Group

In [ ]:
for col in ['gender', 'geography', 'income_proxy']:
    rates = df.groupby(col)['default_label'].mean().sort_values(ascending=False)
    print(f'\nDefault rate by {col}:')
    for k, v in rates.items():
        print(f'  {k:15s}: {v:.2%}')

## 5. UPI Signal Analysis

In [ ]:
# Expand list columns to compute monthly stats
upi_count_df = pd.DataFrame(df['upi_count'].tolist(),
                            columns=[f'm{i+1}' for i in range(12)])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

total_upi_6m = upi_count_df[[f'm{i}' for i in range(7, 13)]].sum(axis=1)
axes[0].hist(total_upi_6m, bins=40, color='#0066FF', edgecolor='#0A0A0A', linewidth=0.8)
axes[0].set_title('Total UPI Txns (6-month window)', fontweight='bold')
axes[0].set_xlabel('Transaction count')
axes[0].set_ylabel('Frequency')

monthly_means = upi_count_df.mean()
axes[1].plot(range(1, 13), monthly_means.values, 'o-', color='#0066FF', linewidth=2, markersize=6)
axes[1].fill_between(range(1, 13), monthly_means.values, alpha=0.2, color='#0066FF')
axes[1].set_title('Mean Monthly UPI Transaction Count', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Mean count')

plt.tight_layout()
plt.savefig('../models/eda_upi.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Income Shock Prevalence

In [ ]:
job_loss_rate = df['income_shock_job_loss'].mean()
health_shock_rate = df['income_shock_health'].mean()
either_shock = (df['income_shock_job_loss'] | df['income_shock_health']).mean()

print(f'Job loss shock prevalence : {job_loss_rate:.2%}')
print(f'Health shock prevalence   : {health_shock_rate:.2%}')
print(f'At least one shock        : {either_shock:.2%}')

# Default rate among shock vs non-shock
shock_mask = df['income_shock_job_loss'] | df['income_shock_health']
print(f'\nDefault rate with shock   : {df[shock_mask]["default_label"].mean():.2%}')
print(f'Default rate without shock: {df[~shock_mask]["default_label"].mean():.2%}')

## 7. MSME vs Non-MSME

In [ ]:
msme_counts = df['is_msme'].value_counts()
print(f'MSME applicants     : {msme_counts.get(True, 0):,} ({msme_counts.get(True, 0)/len(df):.1%})')
print(f'Non-MSME applicants : {msme_counts.get(False, 0):,} ({msme_counts.get(False, 0)/len(df):.1%})')
print(f'\nDefault rate MSME     : {df[df.is_msme]["default_label"].mean():.2%}')
print(f'Default rate non-MSME : {df[~df.is_msme]["default_label"].mean():.2%}')